# 第一个接口

In [7]:
from fastapi import FastAPI

app = FastAPI()

profile = {
    'hello' : '关于我',
    'helloSubTitle':'项目，创意'
}

@app.get('/api/profile')
def get_profile():
    return profile


## 手动启动
使用 `uvicorn fast_api_practice.hello_fast_api:app --reload` 进行启动。
意思是：让 uvicorn 去找 fast_api_practice.hello_fast_api.py 文件里的 app（这里不写 .py）。--reload 是改代码自动重启——我们在前端早就享受过这待遇（npm run dev 改代码即时生效），这是后端的同款，开发时开着它，就不用重启了。
- `uvicorn` ：启动命令
- `fast_api_practice.hello_fast_api:app` ：前者为模块&具体文件名字，用找模块的 `.` 进行分割。后者为具体该路径下的某个实例，即fast_api的实例
- `--reload`：表示支持热重载


## 使用fastapi 命令快捷启动

```shell
fastapi dev .\hello_fast_api.py
```


# 开始交互（编写post接口）

In [ ]:
from pydantic import BaseModel

class WeatherRequest(BaseModel):
    """
    BaseModel 提供校验、类型转换、自动 __init__ 等功能，同时它是 FastAPI识别"这是请求体模型"的依据。不继承就只是个普通类，注解不生效，FastAPI 也不会从 body 解析它。
    """
    area:str

@app.post('/api/post')
# def post_weather(area:str) -> dict :  # 表单形式
def post_weather(req:WeatherRequest) -> dict :  # json形式
    area = req.area
    if area =='北京':
        return {
            "area":area,
            "weather": "晴天",
            "temperature":"38℃"
        }
    elif area == '上海':
        return {
            "area": area,
            "weather": "雨天",
            "temperature": "338℃"
        }
    return {
        "area": area,
        "weather": "服务器繁忙，稍后重试",
        "temperature": "服务器繁忙，稍后重试"
    }

# 参数校验


In [ ]:
from fastapi import Path,Query
from pydantic import Field

class PathValidateReq(BaseModel):
    id:int = Field(5,lt=100,gt=0,description="取值范围必须在0-100之间")
    name:str = Field(...,min_length=1,max_length=5,description="参数字符长度需在1-5之间")

@app.post("/api/validate/{path_var}")
def validate(req:PathValidateReq,path_var:int = Path(...,lt=20,gt=0)):
    return {
        "id":f"id是{req.id}",
        "name": f"name是{req.name}",
        "path_var": f"path_var 是 {path_var}"
    }

## 一、四种参数类型

| 类型 | 标记 | 值从哪来 | 默认值 |
|------|------|---------|--------|
| 路径参数 | `Path()` | URL 路径 `{path_var}` | ❌ 不允许 |
| 查询参数 | `Query()` | URL `?keyword=xxx` | ✅ 允许 |
| 请求体 | 模型字段 `Field()` | JSON body | ✅ 允许 |
| 文件上传 | `File()` | multipart/form-data | 必填 |

## 二、三条核心规则

### 规则 1：位置决定语义

- **函数参数**用 `Path()` / `Query()` / `File()` → 声明"值从哪来"
- **模型字段**一律用 `Field()` → 值永远是请求体 JSON

### 规则 2：默认值口诀

| 默认值 | 含义 |
|--------|------|
| `...` | 必填 |
| `None` | 可选 |
| 具体值 | 带默认值 |

> 路径参数永远必填——URL 缺了它路由就不匹配。

### 规则 3：`Annotated` 推荐写法

```python
def f(
    path_var: Annotated[int, Path(lt=100)],
    page: Annotated[int, Query(ge=1)] = 1,   # 默认值放参数位置
    req: WeatherRequest,                      # 模型直接收 body
    file: Annotated[bytes, File()],
):
    ...
```

## 常见坑
1. `Path()` 写进模型字段 → 断言报错（路径参数不能有默认值）

## 五、一句话记忆
> **模型里用 `Field`，函数里用 `Path` / `Query` / `File`；`...` 必填、`None` 可选。**

# 异常处理

In [ ]:
from fastapi import HTTPException

@app.get("/api/exception")
def get_exception(text:str):
    if text == 'normal':
        return "normal response"
    else:
        return HTTPException(status_code=403,detail="forbidden")

对于客户端引发的错误，应当使用 fastapi.HTTPException 来中断正常处理流程，并返回标准错误响应